In [ ]:
import os
os.environ['CUDA_VISIBLE_DEVICES'] = '0'

from cmm_eval import BatchRecorder, BatchStore, CMMSaturationEvaluator, default_checkpoints, plot_sweep

import torch
from MeMoHF.modelling_memo_tokenizer import MeMoTokenizer
from MeMoHF.modelling_memo_configuration import MeMoConfig
from MeMoHF.modelling_memo import MeMoForCausalLM
from MeMoHF.evaluating_memo import Evaluation
from MeMoHF.utils import seed_everything

seed_everything(42)

In [ ]:
d,h,l = 2048, 4, 5
chunk_length =  (h ** l)

compositionOp='ProdWithShuffling' #'SCC'#'JLT' #

NUMBER_OF_DOCS=100

CLEN_AFTER_RUN = False

In [ ]:
padding_vector_component_values= {
    'ProdWithShuffling': 1/(d**(1/2)),
    'Prod': 1/(d**(1/2)), 
    'Sum': 0,
    'SCC': 'circular_convolution_identity',
    'JLT': 'in_distribution'
}
padding_vector_component_values = padding_vector_component_values[compositionOp]


# Initializing a standard Tokenizer
max_length = chunk_length 
tokenizer = MeMoTokenizer.from_pretrained("EleutherAI/gpt-neox-20b", 
                                          padding_side='left', truncation_side='left', 
                                          model_max_length=max_length, head_number=h)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.pad_token_id = tokenizer.pad_token_id

# Intializing Memo Configuration
config = MeMoConfig(vocab_size=len(tokenizer), #tokenizer.vocab_size, 
    hidden_size=d, 
    num_hidden_layers=l,
    num_attention_heads=h,
    chunk_length=chunk_length,
    bos_token_id=tokenizer.bos_token_id,
    eos_token_id=tokenizer.eos_token_id,
    pad_token_id=tokenizer.pad_token_id,
    compositionOp=compositionOp,#'ProdWithShuffling',
    padding_seq_idx=tokenizer.pad_token_id,
    padding_vector_component_values=padding_vector_component_values #new!
                    
)

# Initializing the Memo Model from the configuration

model = MeMoForCausalLM(config) 
model.training = True
model.train()
print(model)
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)} is available.")
    model.to('cuda')


In [ ]:
class CMM_evaluator:
    def __init__(self, model, tokenizer, dump_dir):
        self.model = model
        self.tokenizer = tokenizer
        self.dump_dir = dump_dir

    def record(self, data):

        rec = BatchRecorder(self.dump_dir, max_batches_per_layer=1000)   # cap to control disk usage
        for l, layer in enumerate(self.model.memo.layers):
            rec.attach(layer, l)

        for i in range(len(data)):
            model.memorize_text(data[i])

        rec.detach_all()

    def evaluate(self, layer_level):
        layer = self.model.memo.layers[layer_level]            # same MeMoLayer config (d, h, compositionOp) as at recording time
        store = BatchStore(self.dump_dir, layer_level)

        ev = CMMSaturationEvaluator(layer, store, n_holdout_batches=2)
        checkpoints = default_checkpoints(ev.max_train_batches, n_points=8)

        rows = ev.sweep(checkpoints, n_eval=2000, n_fresh=2000,
                        out_json=f"{self.dump_dir}/layer{layer_level}_sweep.json")
        plot_sweep(rows, f"{self.dump_dir}/layer{layer_level}_sweep.png")

In [ ]:
format(NUMBER_OF_DOCS, '06d')

In [ ]:
from datasets import Dataset, DatasetDict, Features, Value, load_dataset, load_from_disk, concatenate_datasets

data = load_from_disk(f'LearningEvaluation/training_data/scaling_curve/n={format(NUMBER_OF_DOCS, "06d")}')

tok_data = data.map(lambda x: tokenizer.get_text_batch_encoding(x['text'], truncation=True))
tok_data.set_format(type='torch', columns=['input_ids', 'labels'])
print(tok_data)
tok_data[0]['input_ids'].shape, tok_data[0]['labels'].shape

In [ ]:
dump_dir = f'memo_dump_{compositionOp}_{h}|{l}|{d}_{format(NUMBER_OF_DOCS, "06d")}'


cmm_evaluator = CMM_evaluator(
    model=model, 
    tokenizer=tokenizer, 
    dump_dir=dump_dir
)

cmm_evaluator.record(tok_data)

In [ ]:
for layer_level in range(l):
    print("*"*80)
    print(layer_level)
    print("*"*80)

    cmm_evaluator.evaluate(layer_level)

In [ ]:
import shutil

if CLEN_AFTER_RUN:
    shutil.rmtree(dump_dir)